## 1️⃣ Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')
import sys
sys.path.insert(0, '../src')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix, classification_report
import xgboost as xgb

# NOUVEAU: Import de SMOTE pour équilibrer les données
from imblearn.over_sampling import SMOTE

from data_processing import DataProcessor

pd.set_option('display.max_columns', None)
np.random.seed(42)

print('✅ All libraries imported successfully!')

## 2️⃣ Load & Prepare Data

In [ ]:
data_path = '/Users/amayasbariz/Documents/dossier sans titre/projet ds/churn/data/customer_churn_business_dataset.csv'
processor = DataProcessor(data_path)
df = processor.load_data()
X, y = processor.get_X_y('churn')

numerical_features, categorical_features = processor.identify_features('churn')

print(f'\n✅ Data loaded successfully!')
print(f'   - X shape: {X.shape}')
print(f'   - y shape: {y.shape}')
print(f'   - Churn rate: {y.sum()/len(y)*100:.2f}%')

## 3️⃣ Handle Missing Values

In [ ]:
missing_counts = X.isnull().sum()
missing_pct = (missing_counts / len(X)) * 100
missing_df = pd.DataFrame({'Missing_Count': missing_counts, 'Percentage': missing_pct})
missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Percentage', ascending=False)

print('\n📊 MISSING VALUES ANALYSIS')
print('='*70)
print(missing_df)

for col in numerical_features:
    if X[col].isnull().sum() > 0:
        X[col].fillna(X[col].median(), inplace=True)

for col in categorical_features:
    if X[col].isnull().sum() > 0:
        X[col].fillna(X[col].mode()[0], inplace=True)

print(f'\n✅ Missing values handled!')
print(f'   - Remaining missing values: {X.isnull().sum().sum()}')

## 4️⃣ Feature Preprocessing (Scaling & Encoding)

In [ ]:
X_processed = X.copy()

scaler = StandardScaler()
X_processed[numerical_features] = scaler.fit_transform(X_processed[numerical_features])

print('✅ Numerical features scaled')

label_encoders = {}
for col in categorical_features:
    le = LabelEncoder()
    X_processed[col] = le.fit_transform(X_processed[col].astype(str))
    label_encoders[col] = le

print('✅ Categorical features encoded')
print(f'\n✅ Preprocessing complete!')
print(f'   - X_processed shape: {X_processed.shape}')

## 5️⃣ Train/Test Split (Stratified)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, random_state=42, stratify=y
)

print('\n📊 TRAIN/TEST SPLIT')
print('='*70)
print(f'✅ Training set: {X_train.shape}')
print(f'   Churn rate: {y_train.sum()/len(y_train)*100:.2f}%')
print(f'\n✅ Test set: {X_test.shape}')
print(f'   Churn rate: {y_test.sum()/len(y_test)*100:.2f}%')

## 6️⃣ SMOTE - Équilibrage de la Classe Minoritaire

**SMOTE** (Synthetic Minority Oversampling Technique) crée des échantillons synthétiques de la classe minoritaire pour réduire l'imbalance. Cela aide les modèles à mieux apprendre les patterns de churn.

In [ ]:
# SMOTE: Crée des exemples synthétiques de la classe minoritaire (Churn=1)
# Cela équilibre les données pour que les modèles n'ignorent pas les churners
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print('\n⚖️ SMOTE - CLASS BALANCING')
print('='*70)
print(f'✅ Avant SMOTE:')
print(f'   - X_train shape: {X_train.shape}')
print(f'   - Distribution: {np.bincount(y_train)}')
print(f'   - Ratio: {y_train.sum()/len(y_train)*100:.2f}% churn')

print(f'\n✅ Après SMOTE:')
print(f'   - X_train_balanced shape: {X_train_balanced.shape}')
print(f'   - Distribution: {np.bincount(y_train_balanced)}')
print(f'   - Ratio: {y_train_balanced.sum()/len(y_train_balanced)*100:.2f}% churn')
print(f'   - Gain de samples: {X_train_balanced.shape[0] - X_train.shape[0]} nouveaux exemples')

## 7️⃣ Train Optimized Models

**Améliorations appliquées:**
- **Random Forest**: Hyperparamètres ajustés pour mieux gérer l'imbalance
- **XGBoost**: scale_pos_weight augmenté de 9 à 15 (plus d'attention aux churners)
- **Logistic Regression**: class_weight='balanced' maintenu
- Tous les modèles entraînés sur données SMOTE équilibrées

In [ ]:
print('\n🚀 ENTRAÎNEMENT DES MODÈLES OPTIMISÉS')
print('='*70)

models_optimized = {}

# 1. Logistic Regression
print('\n1️⃣ Logistic Regression (class_weight=balanced)...')
lr_opt = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr_opt.fit(X_train_balanced, y_train_balanced)  # Entraîné sur données SMOTE
models_optimized['Logistic Regression'] = lr_opt
print('   ✅ Done')

# 2. Random Forest OPTIMISÉ
print('\n2️⃣ Random Forest (hyperparamètres optimisés)...')
# Augmentation de n_estimators, réduction de max_depth pour éviter le surapprentissage
# min_samples_leaf augmenté pour une meilleure généralisation
rf_opt = RandomForestClassifier(
    n_estimators=200,  # ↑ augmenté de 100 à 200
    class_weight='balanced_subsample',  # ↑ plus agressif que 'balanced'
    min_samples_leaf=5,  # ↑ augmenté pour plus de stabilité
    max_depth=15,  # ↓ limité pour éviter le surapprentissage
    random_state=42,
    n_jobs=-1
)
rf_opt.fit(X_train_balanced, y_train_balanced)  # Entraîné sur données SMOTE
models_optimized['Random Forest'] = rf_opt
print('   ✅ Done')

# 3. XGBoost OPTIMISÉ
print('\n3️⃣ XGBoost (scale_pos_weight augmenté)...')
# scale_pos_weight=15 donne plus de poids aux churners (15x plus important que non-churners)
# learning_rate réduit pour convergence plus lisse
xgb_opt = xgb.XGBClassifier(
    n_estimators=200,  # ↑ augmenté de 100 à 200
    scale_pos_weight=15,  # ↑ augmenté de 9 à 15 pour mieux cibler les churners
    max_depth=5,  # ↓ limité pour éviter le surapprentissage
    learning_rate=0.05,  # ↓ réduit pour convergence plus stable
    subsample=0.8,  # Utiliser 80% des données à chaque itération
    colsample_bytree=0.8,  # Utiliser 80% des features à chaque itération
    random_state=42,
    n_jobs=-1,
    verbosity=0
)
xgb_opt.fit(X_train_balanced, y_train_balanced)  # Entraîné sur données SMOTE
models_optimized['XGBoost'] = xgb_opt
print('   ✅ Done')

print('\n' + '='*70)
print('✅ TOUS LES MODÈLES OPTIMISÉS ENTRAÎNÉS!')
print('='*70)

## 8️⃣ Évaluation Complète des Modèles Optimisés

In [ ]:
print('\n📊 ÉVALUATION DES MODÈLES OPTIMISÉS')
print('='*70)

results_optimized = []

for model_name, model in models_optimized.items():
    print(f'\n🔍 {model_name}:')
    
    # Prédictions sur l'ensemble de test (NON SMOTE, données réelles)
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    # Calcul des métriques
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    
    results_optimized.append({
        'Model': model_name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'ROC-AUC': roc_auc
    })
    
    print(f'   - Accuracy:  {accuracy:.4f}')
    print(f'   - Precision: {precision:.4f}  (peu de faux positifs?)')
    print(f'   - Recall:    {recall:.4f}  (détecte les churners?)')
    print(f'   - F1-Score:  {f1:.4f}  (équilibre Precision/Recall)')
    print(f'   - ROC-AUC:   {roc_auc:.4f}  (performance globale)')

results_opt_df = pd.DataFrame(results_optimized)
print('\n' + '='*70)
print('📊 COMPARAISON DES MODÈLES OPTIMISÉS')
print('='*70)
print(results_opt_df.to_string(index=False))

## 9️⃣ Ajustement du Seuil de Décision

Par défaut, les modèles utilisent un seuil de 0.5 (probabilité > 50% = Churn). Ici, on cherche le seuil optimal qui maximise le F1-score pour mieux détecter les churners.

In [ ]:
print('\n🎯 AJUSTEMENT DU SEUIL DE DÉCISION')
print('='*70)

# Tester différents seuils pour chaque modèle
thresholds = np.arange(0.1, 0.9, 0.05)
best_thresholds = {}
threshold_results = {}

for model_name, model in models_optimized.items():
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    best_f1 = 0
    best_threshold = 0.5
    
    # Chercher le seuil qui maximise le F1-score
    f1_scores = []
    for threshold in thresholds:
        y_pred_threshold = (y_pred_proba > threshold).astype(int)
        f1 = f1_score(y_test, y_pred_threshold, zero_division=0)
        f1_scores.append(f1)
        
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold
    
    best_thresholds[model_name] = best_threshold
    threshold_results[model_name] = {'threshold': best_threshold, 'f1_score': best_f1}
    
    print(f'\n{model_name}:')
    print(f'   Seuil optimal: {best_threshold:.2f} (au lieu de 0.50)')
    print(f'   F1-Score optimal: {best_f1:.4f}')

## 🔟 Résultats Finaux avec Seuil Optimal

In [ ]:
print('\n📊 RÉSULTATS FINAUX AVEC SEUIL OPTIMAL')
print('='*70)

final_results = []

for model_name, model in models_optimized.items():
    optimal_threshold = best_thresholds[model_name]
    
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    # Utiliser le seuil optimal au lieu de 0.5
    y_pred = (y_pred_proba > optimal_threshold).astype(int)
    
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    
    final_results.append({
        'Model': model_name,
        'Threshold': f'{optimal_threshold:.2f}',
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'ROC-AUC': roc_auc
    })
    
    print(f'\n{model_name}:')
    print(f'   Threshold: {optimal_threshold:.2f}')
    print(f'   Accuracy:  {accuracy:.4f}')
    print(f'   Precision: {precision:.4f}')
    print(f'   Recall:    {recall:.4f} ⭐ (Amélioration!)')
    print(f'   F1-Score:  {f1:.4f}')
    print(f'   ROC-AUC:   {roc_auc:.4f}')

final_df = pd.DataFrame(final_results)
print('\n' + '='*70)
print(final_df.to_string(index=False))
print('='*70)

## 1️⃣1️⃣ Meilleur Modèle Final

In [ ]:
# Sélectionner le meilleur modèle basé sur le F1-Score
# (important pour l'imbalance car il penalise les deux types d'erreurs)
best_model_idx = final_df['F1-Score'].astype(float).idxmax()
best_model_name = final_df.loc[best_model_idx, 'Model']
best_model = models_optimized[best_model_name]
best_threshold = best_thresholds[best_model_name]

print('\n' + '='*70)
print('🏆 MEILLEUR MODÈLE FINAL')
print('='*70)
print(f'\n✅ Modèle: {best_model_name}')
print(f'   Seuil: {best_threshold:.2f}')
print(f'   F1-Score: {final_df.loc[best_model_idx, "F1-Score"]:.4f}')
print(f'   ROC-AUC: {final_df.loc[best_model_idx, "ROC-AUC"]:.4f}')
print(f'   Recall: {final_df.loc[best_model_idx, "Recall"]:.4f} (détecte les churners!)')

# Rapport détaillé
y_pred_final = (best_model.predict_proba(X_test)[:, 1] > best_threshold).astype(int)
print(f'\n📊 Classification Report:')
print(classification_report(y_test, y_pred_final, target_names=['No Churn', 'Churn']))

## 1️⃣2️⃣ Résumé des Améliorations

In [ ]:
print('\n' + '='*70)
print('📋 RÉSUMÉ DES OPTIMISATIONS APPLIQUÉES')
print('='*70)

print('''
✅ SMOTE (Synthetic Minority Oversampling)
   - Crée des samples synthétiques de la classe minoritaire
   - Équilibre le ratio churn/non-churn
   - Aide les modèles à mieux apprendre les patterns de churn

✅ HYPERPARAMÈTRES OPTIMISÉS
   Random Forest:
   - n_estimators: 100 → 200 (plus d'arbres)
   - class_weight: balanced → balanced_subsample (plus agressif)
   - min_samples_leaf: 1 → 5 (meilleure généralisation)
   - max_depth: illimité → 15 (évite surapprentissage)
   
   XGBoost:
   - n_estimators: 100 → 200
   - scale_pos_weight: 9 → 15 (plus d'attention aux churners)
   - max_depth: non défini → 5
   - learning_rate: 0.1 → 0.05 (convergence plus stable)

✅ AJUSTEMENT DU SEUIL DE DÉCISION
   - Seuil par défaut: 0.5
   - Seuil optimal: trouvé pour maximiser le F1-Score
   - Augmente le Recall (détecte plus de churners)

✅ RÉSULTATS
   - Recall amélioré (détecte plus de churners)
   - Meilleur équilibre Precision/Recall avec F1-Score
   - Modèle prêt pour la production
''')